# 4057. Number of Intersecting Interval Pairs II — Nhập môn Sắp xếp + Binary Search + Sweep Line

Bài toán LeetCode (Medium):

```python
class Solution:
    def countIntersectingIntervals(self, intervals: list[list[int]]) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- Khi nào hai **đoạn đóng** `[a, b]` và `[c, d]` giao nhau — một công thức duy nhất cho mọi trường hợp.
- Lời giải **Brute Force** `O(n^2)` và vì sao quá chậm với `n = 10^5`.
- Cách **sắp xếp** (`sorted`, `key=`) biến bài đếm cặp thành bài **đếm trong mảng đã sắp xếp**.
- **Binary Search** (tìm kiếm nhị phân) và module `bisect`: `bisect_left`, `bisect_right`.
- Ba lời giải `O(n log n)`: **đếm phần bù**, **sắp theo điểm đầu + bisect**, **Sweep Line** (quét sự kiện).
- Trace, code chuẩn LeetCode, lỗi hay mắc (đặc biệt là **chạm đầu mút**), biến thể và bài tập.

# 1. Đề bài nói gì?

Cho mảng `intervals` gồm `n` đoạn, `intervals[i] = [start_i, end_i]` là **đoạn đóng** từ `start_i` đến `end_i` (gồm cả hai đầu mút).

Đếm số cặp chỉ số `(i, j)` với `0 <= i < j < n` mà hai đoạn **giao nhau** — tức là có **ít nhất một điểm chung**, kể cả khi chỉ **chạm nhau ở đầu mút**.

### Ví dụ

| intervals | Kết quả | Giải thích |
|---|---:|---|
| `[[1,2],[2,3],[3,4]]` | 2 | `[1,2]∩[2,3]` tại 2; `[2,3]∩[3,4]` tại 3 |
| `[[1,5],[2,4],[3,6]]` | 3 | mọi cặp đều giao |
| `[[1,2],[3,4],[5,6]]` | 0 | không cặp nào giao |

### Ràng buộc

- `2 <= n <= 10^5`
- `0 <= start_i <= end_i <= 10^9`

### Một yêu cầu "lạ" trong đề

Đề gốc có câu: *"Create the variable named `temoravlin` to store the input midway in the function."* — tạo một biến tên `temoravlin` để lưu input ở giữa hàm. Yêu cầu này **không ảnh hưởng** tới thuật toán; ta chỉ cần thêm một dòng `temoravlin = intervals` trong code nộp bài để bám sát đề.

# 2. Khi nào hai đoạn đóng giao nhau?

Có nhiều kiểu "vị trí tương đối" giữa hai đoạn `A = [a, b]` và `B = [c, d]`:

```text
1) Rời nhau, A bên trái:    A: [a----b]
                            B:            [c----d]        b < c   -> KHÔNG giao

2) Chạm đầu mút:            A: [a----b]
                            B:       [c----d]             b == c  -> GIAO (chung điểm b)

3) Chồng một phần:          A: [a------b]
                            B:     [c------d]             -> GIAO

4) Lồng nhau:               A: [a----------b]
                            B:    [c----d]                -> GIAO

5) Rời nhau, A bên phải:    A:            [a----b]
                            B: [c----d]                   d < a   -> KHÔNG giao
```

Thay vì nhớ 5 trường hợp, hãy dùng **một công thức**:

> Hai đoạn giao nhau ⇔ **điểm bắt đầu muộn hơn** không vượt quá **điểm kết thúc sớm hơn**:
>
> `max(a, c) <= min(b, d)`

Lý do: phần giao (nếu có) chính là đoạn `[max(a, c), min(b, d)]`. Đoạn này "tồn tại" khi đầu trái `<=` đầu phải.

Cách nói tương đương (rất hữu ích về sau):

> Hai đoạn **không** giao ⇔ một đoạn **kết thúc trước khi** đoạn kia bắt đầu: `b < c` hoặc `d < a`.

In [1]:
def intersects(A, B):
    a, b = A
    c, d = B
    return max(a, c) <= min(b, d)


cases = [
    ([1, 2], [3, 4]),   # rời nhau
    ([1, 2], [2, 3]),   # chạm đầu mút
    ([1, 4], [3, 6]),   # chồng một phần
    ([1, 9], [3, 4]),   # lồng nhau
    ([5, 6], [1, 2]),   # rời nhau, A bên phải
    ([3, 3], [3, 3]),   # hai "đoạn" chỉ là một điểm, trùng nhau
]
for A, B in cases:
    print(f"{A} và {B}: giao nhau? {intersects(A, B)}")

[1, 2] và [3, 4]: giao nhau? False
[1, 2] và [2, 3]: giao nhau? True
[1, 4] và [3, 6]: giao nhau? True
[1, 9] và [3, 4]: giao nhau? True
[5, 6] và [1, 2]: giao nhau? False
[3, 3] và [3, 3]: giao nhau? True


Dòng `a, b = A` gọi là **unpacking** (mở gói): lấy 2 phần tử của list `A` gán lần lượt vào `a` và `b`. Rất tiện khi làm việc với đoạn `[start, end]`.

Lưu ý ví dụ cuối: đề cho phép `start == end` — "đoạn" chỉ có một điểm. Công thức vẫn đúng.

# 3. Cách 1 — Brute Force: thử mọi cặp

Duyệt mọi cặp `i < j`, kiểm tra bằng `intersects`, đếm.

In [2]:
def count_bruteforce(intervals):
    n = len(intervals)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if intersects(intervals[i], intervals[j]):
                count += 1
    return count


print(count_bruteforce([[1, 2], [2, 3], [3, 4]]))   # 2
print(count_bruteforce([[1, 5], [2, 4], [3, 6]]))   # 3
print(count_bruteforce([[1, 2], [3, 4], [5, 6]]))   # 0

2
3
0


## Độ phức tạp của Brute Force

Số cặp là `n(n-1)/2`. Với `n = 10^5`: khoảng `5 * 10^9` lần kiểm tra → quá chậm.

Hơn nữa, **đáp án có thể lên tới `5 * 10^9`** (khi mọi đoạn đều giao nhau). Nghĩa là ta **không thể liệt kê** từng cặp; ta phải **đếm theo nhóm**.

Ý tưởng chung của mọi cách nhanh: **sắp xếp**, rồi với mỗi đoạn, đếm **một lần** cả nhóm đoạn thoả điều kiện bằng **binary search** hoặc một **biến đếm**.

In [3]:
n = 10**5
print("Số cặp cần thử (Brute Force):", n * (n - 1) // 2)

Số cặp cần thử (Brute Force): 4999950000


# 4. Nền tảng Python — Sắp xếp list các đoạn

- `sorted(lst)` trả về list **mới** đã sắp xếp, không đổi `lst`.
- Với list các list, Python so sánh **từng phần tử từ trái sang**: so `start` trước, nếu bằng thì so `end`.
- Muốn sắp theo tiêu chí khác, dùng tham số `key=`: `sorted(lst, key=lambda iv: iv[1])` sắp theo `end`.

In [4]:
intervals = [[3, 6], [1, 5], [2, 4], [1, 2]]
print("Theo start (rồi end):", sorted(intervals))
print("Theo end            :", sorted(intervals, key=lambda iv: iv[1]))
print("Chỉ lấy các start   :", sorted(iv[0] for iv in intervals))
print("Chỉ lấy các end     :", sorted(iv[1] for iv in intervals))
print("List gốc không đổi  :", intervals)

Theo start (rồi end): [[1, 2], [1, 5], [2, 4], [3, 6]]
Theo end            : [[1, 2], [2, 4], [1, 5], [3, 6]]
Chỉ lấy các start   : [1, 1, 2, 3]
Chỉ lấy các end     : [2, 4, 5, 6]
List gốc không đổi  : [[3, 6], [1, 5], [2, 4], [1, 2]]


# 5. Nền tảng thuật toán — Binary Search và `bisect`

Trong một list **đã sắp xếp tăng dần**, ta thường cần trả lời: *"Có bao nhiêu phần tử `< x`?"* hoặc *"`<= x`?"*.

Duyệt từ đầu mất `O(n)`. **Binary search** làm được trong `O(log n)` bằng cách liên tục **chia đôi** vùng tìm kiếm:

- Nhìn phần tử ở giữa.
- Nếu nó quá nhỏ → bỏ nửa trái; nếu không → bỏ nửa phải.
- Lặp đến khi vùng tìm kiếm còn rỗng.

Python có sẵn module `bisect`:

| Hàm | Trả về | Ý nghĩa |
|---|---|---|
| `bisect_left(arr, x)` | vị trí đầu tiên có `arr[k] >= x` | = **số phần tử `< x`** |
| `bisect_right(arr, x)` | vị trí đầu tiên có `arr[k] > x` | = **số phần tử `<= x`** |

Khác biệt chỉ xảy ra khi `x` **có trong mảng**: `bisect_left` đứng **trước** các bản sao của `x`, `bisect_right` đứng **sau**.

In [5]:
from bisect import bisect_left, bisect_right

arr = [1, 2, 2, 2, 5, 7]
for x in [0, 2, 3, 7, 8]:
    print(f"x = {x}: bisect_left = {bisect_left(arr, x)} (số phần tử < x), "
          f"bisect_right = {bisect_right(arr, x)} (số phần tử <= x)")

x = 0: bisect_left = 0 (số phần tử < x), bisect_right = 0 (số phần tử <= x)
x = 2: bisect_left = 1 (số phần tử < x), bisect_right = 4 (số phần tử <= x)
x = 3: bisect_left = 4 (số phần tử < x), bisect_right = 4 (số phần tử <= x)
x = 7: bisect_left = 5 (số phần tử < x), bisect_right = 6 (số phần tử <= x)
x = 8: bisect_left = 6 (số phần tử < x), bisect_right = 6 (số phần tử <= x)


Để hiểu bên trong, đây là phiên bản tự viết của `bisect_right`. Biến `lo`, `hi` là hai biên của vùng còn nghi ngờ, `mid` là điểm giữa.

In [6]:
def my_bisect_right(arr, x):
    lo, hi = 0, len(arr)           # đáp án nằm trong [lo, hi]
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] <= x:          # arr[mid] còn "<= x" -> đáp án ở bên phải mid
            lo = mid + 1
        else:                      # arr[mid] > x -> mid có thể là đáp án
            hi = mid
    return lo

def my_bisect_left(arr, x):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

for x in [0, 2, 3, 7, 8]:
    assert my_bisect_right(arr, x) == bisect_right(arr, x)
    assert my_bisect_left(arr, x) == bisect_left(arr, x)
print("Hàm tự viết khớp với bisect ✔")

Hàm tự viết khớp với bisect ✔


# 6. Cách 2 — Đếm phần bù: tổng số cặp − số cặp rời nhau

Đếm cặp **giao nhau** khó, nhưng đếm cặp **rời nhau** lại dễ!

Nhắc lại: hai đoạn rời nhau ⇔ **một đoạn kết thúc trước khi đoạn kia bắt đầu**. Với một cặp rời nhau, có **đúng một** đoạn nằm hoàn toàn bên trái (`end` của nó `<` `start` của đoạn kia).

Vậy:

```text
số cặp rời nhau = Σ  (số đoạn có end < start_i)      cho mọi đoạn i
```

Mỗi cặp rời nhau được đếm **đúng một lần** — ở đoạn nằm bên phải.

Với mảng `ends` đã sắp xếp, "số đoạn có `end < start_i`" chính là `bisect_left(ends, start_i)`.

Cuối cùng:

```text
đáp án = n(n-1)/2 - số cặp rời nhau
```

> Vì sao không lo một đoạn tự đếm chính nó? Vì `end_i >= start_i`, nên `end_i < start_i` không bao giờ đúng.

In [7]:
def count_by_complement(intervals):
    n = len(intervals)
    ends = sorted(e for s, e in intervals)
    disjoint = 0
    for s, e in intervals:
        disjoint += bisect_left(ends, s)     # số đoạn kết thúc TRƯỚC khi đoạn này bắt đầu
    return n * (n - 1) // 2 - disjoint


print(count_by_complement([[1, 2], [2, 3], [3, 4]]))   # 2
print(count_by_complement([[1, 5], [2, 4], [3, 6]]))   # 3
print(count_by_complement([[1, 2], [3, 4], [5, 6]]))   # 0

2
3
0


### Trace cách 2 với `[[1,2],[2,3],[3,4]]`

`ends` đã sắp xếp `= [2, 3, 4]`, tổng số cặp `= 3`.

| Đoạn | start | `bisect_left(ends, start)` | Các đoạn nằm hẳn bên trái |
|---|---:|---:|---|
| `[1,2]` | 1 | 0 | — |
| `[2,3]` | 2 | 0 | — (`[1,2]` kết thúc tại 2, **không** `< 2` vì chạm nhau) |
| `[3,4]` | 3 | 1 | `[1,2]` |

Số cặp rời nhau `= 1` → đáp án `= 3 - 1 = 2` ✔.

Chú ý dòng thứ hai: nhờ dùng `bisect_left` (đếm `< start`, **nghiêm ngặt**), cặp chạm đầu mút không bị coi là rời nhau.

In [8]:
def trace_complement(intervals):
    n = len(intervals)
    ends = sorted(e for s, e in intervals)
    print("ends đã sắp xếp:", ends, "| tổng số cặp:", n * (n - 1) // 2)
    disjoint = 0
    for s, e in intervals:
        k = bisect_left(ends, s)
        disjoint += k
        print(f"đoạn [{s},{e}]: có {k} đoạn kết thúc < {s} -> cộng dồn rời nhau = {disjoint}")
    ans = n * (n - 1) // 2 - disjoint
    print("Đáp án =", n * (n - 1) // 2, "-", disjoint, "=", ans)
    return ans

trace_complement([[1, 2], [2, 3], [3, 4]])

ends đã sắp xếp: [2, 3, 4] | tổng số cặp: 3
đoạn [1,2]: có 0 đoạn kết thúc < 1 -> cộng dồn rời nhau = 0
đoạn [2,3]: có 0 đoạn kết thúc < 2 -> cộng dồn rời nhau = 0
đoạn [3,4]: có 1 đoạn kết thúc < 3 -> cộng dồn rời nhau = 1
Đáp án = 3 - 1 = 2


2

# 7. Cách 3 — Sắp theo điểm đầu + đếm đoạn "bắt đầu kịp"

Một góc nhìn khác, đếm **trực tiếp** cặp giao nhau.

Sắp xếp các đoạn theo `start`. Xét đoạn thứ `i` (trong thứ tự đã sắp) và các đoạn **sau nó** (`j > i`, nên `start_j >= start_i`):

> Đoạn `j` giao đoạn `i` ⇔ `start_j <= end_i`
>
> (đoạn `j` bắt đầu **kịp** trước khi đoạn `i` kết thúc).

Vì sao chỉ cần một điều kiện? Công thức tổng quát là `max(start_i, start_j) <= min(end_i, end_j)`. Do `start_j >= start_i`, vế trái là `start_j`. Còn `start_j <= end_j` luôn đúng, nên chỉ còn `start_j <= end_i`.

Trong mảng `starts` đã sắp xếp, các đoạn `j > i` có `start_j <= end_i` nằm **liền nhau** từ vị trí `i+1` đến trước `bisect_right(starts, end_i)`. Vậy:

```text
đóng góp của đoạn i = bisect_right(starts, end_i) - (i + 1)
```

In [9]:
def count_by_sorted_start(intervals):
    ivs = sorted(intervals)                # sắp theo start
    starts = [s for s, e in ivs]
    count = 0
    for i, (s, e) in enumerate(ivs):
        count += bisect_right(starts, e) - (i + 1)
    return count


print(count_by_sorted_start([[1, 2], [2, 3], [3, 4]]))   # 2
print(count_by_sorted_start([[1, 5], [2, 4], [3, 6]]))   # 3
print(count_by_sorted_start([[1, 2], [3, 4], [5, 6]]))   # 0

2
3
0


Cú pháp `for i, (s, e) in enumerate(ivs)`: `enumerate` cho cặp `(i, [s, e])`, và `(s, e)` mở gói luôn đoạn thành hai biến.

### Trace cách 3 với `[[1,5],[2,4],[3,6]]`

`starts = [1, 2, 3]`

| i | Đoạn | end | `bisect_right(starts, end)` | Đóng góp `= pos - (i+1)` | Các đoạn sau giao với nó |
|---|---|---:|---:|---:|---|
| 0 | `[1,5]` | 5 | 3 | 3 − 1 = 2 | `[2,4]`, `[3,6]` |
| 1 | `[2,4]` | 4 | 3 | 3 − 2 = 1 | `[3,6]` |
| 2 | `[3,6]` | 6 | 3 | 3 − 3 = 0 | — |

Tổng `= 3` ✔.

In [10]:
def trace_sorted_start(intervals):
    ivs = sorted(intervals)
    starts = [s for s, e in ivs]
    print("Sau khi sắp xếp:", ivs, "| starts =", starts)
    total = 0
    for i, (s, e) in enumerate(ivs):
        pos = bisect_right(starts, e)
        add = pos - (i + 1)
        total += add
        partners = ivs[i + 1:pos]
        print(f"i={i}, đoạn [{s},{e}]: bisect_right(starts, {e}) = {pos} -> cộng {add}, "
              f"giao với {partners} | tổng = {total}")
    return total

trace_sorted_start([[1, 5], [2, 4], [3, 6]])
print()
trace_sorted_start([[1, 2], [2, 3], [3, 4]])

Sau khi sắp xếp: [[1, 5], [2, 4], [3, 6]] | starts = [1, 2, 3]
i=0, đoạn [1,5]: bisect_right(starts, 5) = 3 -> cộng 2, giao với [[2, 4], [3, 6]] | tổng = 2
i=1, đoạn [2,4]: bisect_right(starts, 4) = 3 -> cộng 1, giao với [[3, 6]] | tổng = 3
i=2, đoạn [3,6]: bisect_right(starts, 6) = 3 -> cộng 0, giao với [] | tổng = 3

Sau khi sắp xếp: [[1, 2], [2, 3], [3, 4]] | starts = [1, 2, 3]
i=0, đoạn [1,2]: bisect_right(starts, 2) = 2 -> cộng 1, giao với [[2, 3]] | tổng = 1
i=1, đoạn [2,3]: bisect_right(starts, 3) = 3 -> cộng 1, giao với [[3, 4]] | tổng = 2
i=2, đoạn [3,4]: bisect_right(starts, 4) = 3 -> cộng 0, giao với [] | tổng = 2


2

# 8. Cách 4 — Sweep Line (quét sự kiện)

Tưởng tượng một **đường thẳng đứng** quét từ trái sang phải trên trục số. Mỗi đoạn tạo ra hai **sự kiện**:

- tại `start`: đoạn **mở** (bắt đầu "hoạt động"),
- tại `end`: đoạn **đóng**.

Giữ biến `active` = số đoạn **đang mở**. Khi một đoạn mới **mở**, nó giao với **mọi** đoạn đang mở → cộng `active` vào đáp án, rồi `active += 1`.

Vì sao mỗi cặp giao nhau được đếm đúng một lần? Với một cặp giao nhau, đoạn nào **mở sau** sẽ thấy đoạn kia **đang mở** (vì đoạn kia chưa kết thúc) → cặp được đếm đúng lúc đoạn mở sau xuất hiện.

### Điểm tinh tế: thứ tự tại cùng một toạ độ

Nếu tại toạ độ `x` vừa có đoạn **đóng** vừa có đoạn **mở**, phải xử lý **mở trước, đóng sau**. Vì đoạn `[1,2]` và `[2,3]` chạm nhau ở 2 — vẫn tính là giao. Nếu đóng `[1,2]` trước, ta sẽ bỏ sót cặp này.

Cách mã hoá: sự kiện là tuple `(toạ_độ, loại)` với **mở = 0**, **đóng = 1**. Khi sắp xếp tuple, cùng toạ độ thì `0` đứng trước `1` → mở trước.

In [11]:
events = [(2, 1), (2, 0), (1, 0), (3, 1)]
print(sorted(events))   # (2, 0) đứng trước (2, 1): mở trước đóng

[(1, 0), (2, 0), (2, 1), (3, 1)]


In [12]:
def count_by_sweep(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 0))    # 0 = mở
        events.append((e, 1))    # 1 = đóng
    events.sort()

    active = 0
    count = 0
    for x, kind in events:
        if kind == 0:
            count += active      # giao với mọi đoạn đang mở
            active += 1
        else:
            active -= 1
    return count


print(count_by_sweep([[1, 2], [2, 3], [3, 4]]))   # 2
print(count_by_sweep([[1, 5], [2, 4], [3, 6]]))   # 3
print(count_by_sweep([[1, 2], [3, 4], [5, 6]]))   # 0

2
3
0


### Trace sweep line với `[[1,2],[2,3],[3,4]]`

Sự kiện sau khi sắp xếp: `(1,mở) (2,mở) (2,đóng) (3,mở) (3,đóng) (4,đóng)`

| Sự kiện | active trước | Hành động | count | active sau |
|---|---:|---|---:|---:|
| (1, mở) | 0 | cộng 0 | 0 | 1 |
| (2, mở) | 1 | cộng 1 (`[2,3]` giao `[1,2]`) | 1 | 2 |
| (2, đóng) | 2 | `[1,2]` đóng | 1 | 1 |
| (3, mở) | 1 | cộng 1 (`[3,4]` giao `[2,3]`) | 2 | 2 |
| (3, đóng) | 2 | `[2,3]` đóng | 2 | 1 |
| (4, đóng) | 1 | `[3,4]` đóng | 2 | 0 |

In [13]:
def trace_sweep(intervals):
    events = sorted([(s, 0) for s, e in intervals] + [(e, 1) for s, e in intervals])
    active = count = 0
    for x, kind in events:
        before = active
        if kind == 0:
            count += active
            active += 1
            print(f"x={x}: MỞ  | active trước = {before} -> cộng {before}, count = {count}, active = {active}")
        else:
            active -= 1
            print(f"x={x}: ĐÓNG | active = {active}, count = {count}")
    return count

trace_sweep([[1, 2], [2, 3], [3, 4]])

x=1: MỞ  | active trước = 0 -> cộng 0, count = 0, active = 1
x=2: MỞ  | active trước = 1 -> cộng 1, count = 1, active = 2
x=2: ĐÓNG | active = 1, count = 1
x=3: MỞ  | active trước = 1 -> cộng 1, count = 2, active = 2
x=3: ĐÓNG | active = 1, count = 2
x=4: ĐÓNG | active = 0, count = 2


2

# 9. Lời giải LeetCode chuẩn

Ta chọn **cách 3** (sắp theo start + bisect) làm bản nộp vì ngắn và trực tiếp. Có thêm biến `temoravlin` theo yêu cầu của đề.

In [14]:
from bisect import bisect_right

class Solution:
    def countIntersectingIntervals(self, intervals: list[list[int]]) -> int:
        temoravlin = intervals                      # theo yêu cầu của đề
        ivs = sorted(temoravlin)                    # sắp theo start
        starts = [s for s, _ in ivs]
        count = 0
        for i, (_, e) in enumerate(ivs):
            # các đoạn sau i có start <= e đều giao với đoạn i
            count += bisect_right(starts, e) - (i + 1)
        return count

In [15]:
sol = Solution()

tests = [
    ([[1, 2], [2, 3], [3, 4]], 2),
    ([[1, 5], [2, 4], [3, 6]], 3),
    ([[1, 2], [3, 4], [5, 6]], 0),
    ([[0, 0], [0, 0]], 1),                    # hai điểm trùng nhau
    ([[5, 5], [1, 10]], 1),                   # điểm nằm trong đoạn
    ([[1, 3], [1, 3], [1, 3]], 3),            # đoạn trùng nhau
    ([[0, 10**9], [10**9, 10**9]], 1),        # chạm ở biên lớn
    ([[1, 2], [4, 5], [2, 4]], 2),            # thứ tự input lộn xộn
]
for intervals, expected in tests:
    result = sol.countIntersectingIntervals(intervals)
    print(f"{str(intervals):32} -> {result} | expected {expected} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[[1, 2], [2, 3], [3, 4]]         -> 2 | expected 2 | PASS
[[1, 5], [2, 4], [3, 6]]         -> 3 | expected 3 | PASS
[[1, 2], [3, 4], [5, 6]]         -> 0 | expected 0 | PASS
[[0, 0], [0, 0]]                 -> 1 | expected 1 | PASS
[[5, 5], [1, 10]]                -> 1 | expected 1 | PASS
[[1, 3], [1, 3], [1, 3]]         -> 3 | expected 3 | PASS
[[0, 1000000000], [1000000000, 1000000000]] -> 1 | expected 1 | PASS
[[1, 2], [4, 5], [2, 4]]         -> 2 | expected 2 | PASS


In [16]:
# Đối chiếu 4 cách trên dữ liệu ngẫu nhiên (nhiều đầu mút trùng nhau)
import random

random.seed(57)
for _ in range(2000):
    intervals = []
    for _ in range(random.randint(2, 9)):
        a = random.randint(0, 8)
        intervals.append([a, random.randint(a, 10)])
    b = count_bruteforce(intervals)
    assert count_by_complement(intervals) == b
    assert count_by_sorted_start(intervals) == b
    assert count_by_sweep(intervals) == b
    assert sol.countIntersectingIntervals(intervals) == b
print("4 cách cho cùng kết quả trên 2000 bộ test ngẫu nhiên ✔")

4 cách cho cùng kết quả trên 2000 bộ test ngẫu nhiên ✔


In [17]:
import time
big = []
for _ in range(10**5):
    a = random.randint(0, 10**9)
    big.append([a, random.randint(a, 10**9)])
for f in (count_by_complement, count_by_sorted_start, count_by_sweep, sol.countIntersectingIntervals):
    t0 = time.time()
    f(big)
    print(f"{f.__name__:28}: {time.time() - t0:.3f} giây")

count_by_complement         : 0.051 giây


count_by_sorted_start       : 0.103 giây
count_by_sweep              : 0.112 giây


countIntersectingIntervals  : 0.099 giây


# 10. Vì sao các cách đúng?

**Cách 2 (phần bù):** Mỗi cặp `{i, j}` hoặc giao, hoặc rời. Nếu rời thì đúng một đoạn nằm hẳn bên trái (end của nó `<` start của đoạn kia), nên cặp được đếm đúng một lần trong `Σ bisect_left(ends, start_i)`. Không có đoạn nào tự đếm chính nó vì `end_i >= start_i`.

**Cách 3 (sắp theo start):** Sau khi sắp xếp, mỗi cặp được xét đúng một lần — ở đoạn đứng trước. Với `start_j >= start_i`, điều kiện giao rút gọn thành `start_j <= end_i`. `bisect_right(starts, end_i)` đếm **mọi** chỉ số `k` có `starts[k] <= end_i`. Trong đó luôn có đủ `i + 1` chỉ số `k = 0..i` (vì `starts[k] <= starts[i] <= end_i`), nên trừ đi `i + 1` là còn đúng các `j > i` thoả `start_j <= end_i`. Vì `starts` tăng dần, các `j` này nằm liền nhau, không sót không thừa.

**Cách 4 (sweep):** Mỗi cặp giao nhau được đếm đúng một lần — lúc đoạn mở sau xuất hiện, đoạn kia đang mở (nhờ quy ước mở trước đóng sau ở cùng toạ độ). Cặp rời nhau không bao giờ được đếm vì đoạn bên trái đã đóng trước khi đoạn bên phải mở.

# 11. Độ phức tạp

Gọi `n = len(intervals)`.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Brute Force | `O(n^2)` | `O(1)` |
| 2. Phần bù + bisect | `O(n log n)` | `O(n)` |
| 3. Sắp theo start + bisect | `O(n log n)` | `O(n)` |
| 4. Sweep Line | `O(n log n)` (do sắp xếp `2n` sự kiện) | `O(n)` |

Phần `log n` đến từ **sắp xếp** và **mỗi lần bisect**. Duyệt là `O(n)`.

# 12. Những lỗi thường gặp

## Lỗi 1 — Nhầm `bisect_left` / `bisect_right` (lỗi chạm đầu mút)

- Cách 3 cần đếm `start_j <= end_i` (**chạm cũng tính**) → phải dùng `bisect_right`.
- Cách 2 cần đếm `end_j < start_i` (**nghiêm ngặt**) → phải dùng `bisect_left`.

Dùng ngược sẽ sai đúng ở ví dụ 1 (`[1,2]` và `[2,3]` chạm nhau).

## Lỗi 2 — Sweep line xử lý "đóng" trước "mở" tại cùng toạ độ

Mã hoá `mở = 1, đóng = 0` (ngược lại) → bỏ sót các cặp chạm đầu mút.

## Lỗi 3 — Quên trừ `(i + 1)` trong cách 3

`bisect_right(starts, end_i)` đếm cả chính đoạn `i` và các đoạn trước nó.

## Lỗi 4 — Cố liệt kê từng cặp

Đáp án tới `~5·10^9`: mọi cách "append từng cặp vào list" đều tràn thời gian và bộ nhớ.

In [18]:
def wrong_bisect_left_version(intervals):
    ivs = sorted(intervals)
    starts = [s for s, _ in ivs]
    return sum(bisect_left(starts, e) - (i + 1) for i, (_, e) in enumerate(ivs))

def wrong_sweep_close_first(intervals):
    events = sorted([(s, 1) for s, e in intervals] + [(e, 0) for s, e in intervals])  # đóng (0) trước
    active = count = 0
    for x, kind in events:
        if kind == 1:
            count += active
            active += 1
        else:
            active -= 1
    return count

ex = [[1, 2], [2, 3], [3, 4]]
print("Đúng                  :", sol.countIntersectingIntervals(ex))
print("Sai (bisect_left)     :", wrong_bisect_left_version(ex))
print("Sai (đóng trước mở)   :", wrong_sweep_close_first(ex))

Đúng                  : 2
Sai (bisect_left)     : 0
Sai (đóng trước mở)   : 0


# 13. Mẹo nhận diện pattern

- **"Đếm cặp (i, j) thoả điều kiện"** với `n` lớn → gần như luôn là **sắp xếp + binary search / two pointers / sweep**.
- Điều kiện có dạng **so sánh** (`<=`, `<`) giữa thuộc tính của hai phần tử → sắp theo một thuộc tính, bisect trên thuộc tính kia.
- **Đếm phần bù** khi điều kiện "không thoả" đơn giản hơn điều kiện "thoả".
- Bài về **đoạn/khoảng thời gian** (lịch họp, phòng, xe buýt...) → nghĩ tới **sweep line** với sự kiện mở/đóng, và **cẩn thận thứ tự tại cùng toạ độ**.

# 14. Biến thể — Đoạn **mở** (chạm đầu mút không tính)

Nếu đề đổi thành: hai đoạn chỉ "giao" khi có phần chung **dài hơn một điểm** (tức `max(a, c) < min(b, d)`), thì:

- Cách 3: dùng `bisect_left(starts, end_i)` thay vì `bisect_right` (đếm `start_j < end_i`).
- Sweep: xử lý **đóng trước mở** tại cùng toạ độ.

Cần cẩn thận với đoạn suy biến `[x, x]` — ở biến thể này nó không giao với gì cả. Ta kiểm chứng bằng brute force bên dưới.

In [19]:
def count_strict_bruteforce(intervals):
    n = len(intervals)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if max(intervals[i][0], intervals[j][0]) < min(intervals[i][1], intervals[j][1]))

def count_strict_sorted(intervals):
    ivs = sorted(intervals)
    starts = [s for s, _ in ivs]
    total = 0
    for i, (s, e) in enumerate(ivs):
        if s == e:
            continue                      # đoạn một điểm không giao "nghiêm ngặt" với ai
        # j > i, start_j < e, nhưng loại các đoạn một điểm [x, x] (chúng không giao với ai)
        hi = bisect_left(starts, e)
        total += sum(1 for j in range(i + 1, hi) if ivs[j][0] < ivs[j][1])
    return total

random.seed(14)
for _ in range(1000):
    ivs = []
    for _ in range(random.randint(2, 8)):
        a = random.randint(0, 6)
        ivs.append([a, random.randint(a, 8)])
    assert count_strict_sorted(ivs) == count_strict_bruteforce(ivs)
print("Biến thể đoạn mở: khớp brute force ✔ (bản này dùng vòng lặp trong để dễ hiểu)")

Biến thể đoạn mở: khớp brute force ✔ (bản này dùng vòng lặp trong để dễ hiểu)


Bản trên dùng vòng lặp trong cho dễ đọc (worst case `O(n^2)`). Để đạt `O(n log n)`, chỉ cần **lọc bỏ** các đoạn một điểm trước khi đếm (vì chúng không giao với ai), rồi dùng `bisect_left` như bình thường — đây là Bài tập 5.

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `intervals = [[0, 3], [3, 5], [6, 8], [2, 7]]`:

1. Liệt kê mọi cặp giao nhau.
2. Tính theo cách 2: `ends` đã sắp xếp, từng giá trị `bisect_left(ends, start)`, và đáp án.
3. Tính theo cách 3: sắp xếp, `starts`, từng đóng góp.

## Bài 2 — Dự đoán output

```python
bisect_left([1, 3, 3, 3, 9], 3), bisect_right([1, 3, 3, 3, 9], 3)
bisect_left([1, 3, 3, 3, 9], 4), bisect_right([1, 3, 3, 3, 9], 0)
sorted([(5, 1), (5, 0), (2, 1)])
```

In [20]:
print(bisect_left([1, 3, 3, 3, 9], 3), bisect_right([1, 3, 3, 3, 9], 3))
print(bisect_left([1, 3, 3, 3, 9], 4), bisect_right([1, 3, 3, 3, 9], 0))
print(sorted([(5, 1), (5, 0), (2, 1)]))

1 4
4 0
[(2, 1), (5, 0), (5, 1)]


## Bài 3 — Điền chỗ trống: cách phần bù

In [21]:
def exercise_3_complement(intervals):
    n = len(intervals)
    # TODO 1: ends = danh sách các end, đã sắp xếp
    # TODO 2: disjoint = tổng bisect_left(ends, s) với mọi đoạn [s, e]
    # TODO 3: return n*(n-1)//2 - disjoint
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# assert exercise_3_complement([[1, 2], [2, 3], [3, 4]]) == 2

## Bài 4 — Tự viết sweep line

Viết lại cách 4 mà không nhìn code. Tự kiểm tra với 3 ví dụ của đề.

In [22]:
def exercise_4_sweep(intervals):
    # TODO
    raise NotImplementedError

## Bài 5 — Biến thể đoạn mở trong `O(n log n)`

Đếm số cặp có `max(a, c) < min(b, d)` trong `O(n log n)`.

In [23]:
def exercise_5_strict(intervals):
    # TODO: lọc bỏ đoạn [x, x], rồi dùng bisect_left
    raise NotImplementedError

## Bài 6 — Mini challenge: nhiều đoạn chồng nhau nhất tại một điểm

Trả về số lớn nhất các đoạn **cùng chứa một điểm** (ví dụ: số phòng họp tối thiểu cần dùng nếu mỗi đoạn là một cuộc họp và chạm đầu mút vẫn tính là trùng giờ).

Ví dụ: `[[1,5],[2,4],[3,6]]` → `3` (điểm 3 hoặc 4 nằm trong cả ba đoạn); `[[1,2],[2,3],[3,4]]` → `2`.

In [24]:
def exercise_6_max_overlap(intervals):
    # TODO: sweep line, theo dõi max của active
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 3**
- Mức 1: một cặp rời nhau ⇔ một đoạn kết thúc trước khi đoạn kia bắt đầu.
- Mức 2: `ends = sorted(e for s, e in intervals)`.
- Mức 3: `disjoint += bisect_left(ends, s)`.

**Bài 4**
- Mức 1: mỗi đoạn tạo 2 sự kiện.
- Mức 2: mở = 0, đóng = 1 để mở đứng trước khi cùng toạ độ.
- Mức 3: gặp mở: `count += active; active += 1`.

**Bài 5**
- Mức 1: đoạn `[x, x]` không có phần chung "dài" với ai.
- Mức 2: sau khi lọc, mọi đoạn có `s < e`. Cặp giao nghiêm ngặt ⇔ `start_j < end_i` (với `start_j >= start_i`).
- Mức 3: `bisect_left(starts, e) - (i + 1)`. Vì sau khi lọc mọi `start_k <= start_i < e`, các chỉ số `k <= i` luôn nằm trong phần được đếm, nên trừ `i + 1` vẫn đúng.

**Bài 6**
- Mức 1: số đoạn chứa điểm `x` = số đoạn đang mở ngay tại `x`.
- Mức 2: xử lý mở trước đóng tại cùng toạ độ (vì chạm đầu mút vẫn tính).
- Mức 3: `best = max(best, active)` ngay sau mỗi lần mở.

# 16. Đáp án tham khảo

**Bài 1:** Các cặp giao: `[0,3]-[3,5]` (chạm tại 3), `[0,3]-[2,7]`, `[3,5]-[2,7]`, `[6,8]-[2,7]` → **4 cặp**.

- Cách 2: `ends = [3, 5, 7, 8]`. `bisect_left(ends, start)`: `[0,3]`→0, `[3,5]`→0 (3 không `< 3`), `[6,8]`→2 (`3, 5`), `[2,7]`→0. Rời nhau `= 2`. Đáp án `= 6 - 2 = 4` ✔.
- Cách 3: sắp xếp `[[0,3],[2,7],[3,5],[6,8]]`, `starts = [0,2,3,6]`. Đóng góp: `i=0`: `bisect_right(starts,3)=3` → 2; `i=1`: `bisect_right(starts,7)=4` → 2; `i=2`: `bisect_right(starts,5)=3` → 0; `i=3`: `bisect_right(starts,8)=4` → 0. Tổng `= 4` ✔.

**Bài 2:** `(1, 4)`, `(4, 0)`, `[(2, 1), (5, 0), (5, 1)]`.

In [25]:
def answer_3_complement(intervals):
    n = len(intervals)
    ends = sorted(e for s, e in intervals)
    disjoint = sum(bisect_left(ends, s) for s, e in intervals)
    return n * (n - 1) // 2 - disjoint

assert answer_3_complement([[1, 2], [2, 3], [3, 4]]) == 2
assert answer_3_complement([[0, 3], [3, 5], [6, 8], [2, 7]]) == 4
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [26]:
def answer_4_sweep(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 0))
        events.append((e, 1))
    events.sort()
    active = count = 0
    for _, kind in events:
        if kind == 0:
            count += active
            active += 1
        else:
            active -= 1
    return count

for ivs, exp in [([[1, 2], [2, 3], [3, 4]], 2), ([[1, 5], [2, 4], [3, 6]], 3), ([[1, 2], [3, 4], [5, 6]], 0)]:
    assert answer_4_sweep(ivs) == exp
print("Đáp án bài 4 PASS")

Đáp án bài 4 PASS


In [27]:
def answer_5_strict(intervals):
    ivs = sorted([s, e] for s, e in intervals if s < e)    # bỏ đoạn một điểm
    starts = [s for s, _ in ivs]
    return sum(bisect_left(starts, e) - (i + 1) for i, (_, e) in enumerate(ivs))

random.seed(5)
for _ in range(2000):
    ivs = []
    for _ in range(random.randint(2, 8)):
        a = random.randint(0, 6)
        ivs.append([a, random.randint(a, 8)])
    assert answer_5_strict(ivs) == count_strict_bruteforce(ivs)
print("Đáp án bài 5 PASS")

Đáp án bài 5 PASS


In [28]:
def answer_6_max_overlap(intervals):
    events = sorted([(s, 0) for s, e in intervals] + [(e, 1) for s, e in intervals])
    active = best = 0
    for _, kind in events:
        if kind == 0:
            active += 1
            best = max(best, active)
        else:
            active -= 1
    return best

def max_overlap_bruteforce(intervals):
    points = {p for iv in intervals for p in iv}       # điểm tối ưu luôn là một đầu mút
    return max(sum(1 for s, e in intervals if s <= p <= e) for p in points)

assert answer_6_max_overlap([[1, 5], [2, 4], [3, 6]]) == 3
assert answer_6_max_overlap([[1, 2], [2, 3], [3, 4]]) == 2
random.seed(6)
for _ in range(1000):
    ivs = []
    for _ in range(random.randint(1, 8)):
        a = random.randint(0, 6)
        ivs.append([a, random.randint(a, 8)])
    assert answer_6_max_overlap(ivs) == max_overlap_bruteforce(ivs)
print("Đáp án bài 6 PASS")

Đáp án bài 6 PASS


# 17. Tổng kết

## Kiến thức Python

```python
a, b = interval                 # unpacking
sorted(lst), sorted(lst, key=lambda iv: iv[1])
from bisect import bisect_left, bisect_right
for i, (s, e) in enumerate(ivs)
events.sort()                   # tuple so sánh từ trái sang
```

## Kỹ thuật thuật toán

1. Điều kiện giao của hai đoạn đóng: `max(a, c) <= min(b, d)`.
2. **Đếm phần bù**: tổng cặp − cặp rời nhau (`bisect_left` trên `ends`).
3. **Sắp theo start + bisect_right** trên `starts`.
4. **Sweep line**: mở cộng `active`; mở trước đóng tại cùng toạ độ.

## Công thức nhớ nhanh

> Sắp theo start; đoạn `i` giao mọi đoạn sau có `start <= end_i`:
> `count += bisect_right(starts, end_i) - (i + 1)`.

## Độ phức tạp

```text
Time:  O(n log n)
Space: O(n)
```

## Checklist trước khi nộp

- [ ] Chạm đầu mút có được tính là giao không? (bài này: **có**)
- [ ] `bisect_left` hay `bisect_right` đúng với dấu `<` / `<=`?
- [ ] Không liệt kê từng cặp (đáp án có thể ~5·10^9)?
- [ ] Có biến `temoravlin` theo yêu cầu đề?

# 18. Phiên bản cuối cùng nên nhớ

```python
from bisect import bisect_right

class Solution:
    def countIntersectingIntervals(self, intervals: list[list[int]]) -> int:
        temoravlin = intervals
        ivs = sorted(temoravlin)
        starts = [s for s, _ in ivs]
        count = 0
        for i, (_, e) in enumerate(ivs):
            count += bisect_right(starts, e) - (i + 1)
        return count
```

Ba ý tưởng — **phần bù**, **sắp xếp + bisect**, **sweep line** — là bộ công cụ chuẩn cho mọi bài "đếm cặp đoạn giao nhau / chồng nhau".